# 6.4习题参考解答代码

这是对手算与本次保存数据的独立核验，不重新训练网络。须使用Python 3.12.x，当前目录为提交根目录。不创建任何文件夹。

In [1]:
import sys, json, math
from pathlib import Path
import numpy as np
assert sys.version_info[:2]==(3,12)
DATA=Path('数据')
assert DATA.is_dir()
P=json.loads((DATA/'A1组-6.4-综合结果.json').read_text(encoding='utf-8'))
print('实际Python版本:',sys.version.split()[0])
print('读取相对路径成功；保存的主实验Python:',P['environment']['python'])

实际Python版本: 3.12.14
读取相对路径成功；保存的主实验Python: 3.12.14


## 题1—5：显式差分、FEM插值、反例与读表

不重新使用主实验的带状求解器，直接以离散正弦公式核验。题5的时间是题设假设，不混作实测。

In [2]:
n=50;h=1/n;x=np.arange(n+1)*h
c=np.pi**2/(4/h**2*np.sin(np.pi*h/2)**2);U=c*np.sin(np.pi*x)
res=(-U[:-2]+2*U[1:-1]-U[2:])/h**2-np.pi**2*np.sin(np.pi*x[1:-1])
assert np.max(abs(res))<1e-10
mid=(x[:-1]+x[1:])/2
err=np.sin(np.pi*mid)-(np.sin(np.pi*x[:-1])+np.sin(np.pi*x[1:]))/2
assert np.allclose(err,np.sin(np.pi*mid)*(1-np.cos(np.pi*h/2)))
z,w=np.polynomial.legendre.leggauss(8);z=(z+1)/2;w=w/2
assert abs(np.sqrt(np.sum(w*(3*z+2)**2)/.5)-np.sqrt(26))<1e-12
for method in ['FD','FEM']:
    rows=[r for r in P['forward']['grid']if r['method']==method and r['n'] in [100,200]]
    print(method,'观测阶',np.log2(rows[0]['rel_l2']/rows[1]['rel_l2']))
for r in P['forward']['summary']:print(r)
print('题5严格整数门槛:',math.floor(1200/(.2-.002))+1,math.floor((1200-100)/(.005-.002))+1)
print('题1—5核验通过')

FD 观测阶 2.000127127457044
FEM 观测阶 1.9999766916702284
{'method': 'PINN-soft', 'seeds': 3, 'l2_min': 1.0683778276008243e-05, 'l2_median': 0.00010761519976393324, 'l2_max': 0.00022445091438356795, 'time_median_s': 4.936952000000019}
{'method': 'PINN-hard', 'seeds': 3, 'l2_min': 2.5249412110193524e-06, 'l2_median': 2.948492045787438e-06, 'l2_max': 5.4295319705945265e-06, 'time_median_s': 5.737410499998077}
题5严格整数门槛: 6061 366667
题1—5核验通过


## 题6：参数信息与最小二乘

用独立的公式检查无噪声内点可辨识、端点无信息，以及双参数同幅值反例。

In [3]:
for label,xd in [('E1',np.arange(1,6)/6),('E3',np.arange(1,6)/100)]:
    q=np.sin(np.pi*xd);y=q/2;S=q@q;a=q@y/S
    assert abs(1/a-2)<1e-12
    print(label,'信息量S=',S,'无噪声kappa=',1/a)
q_boundary=np.zeros(2);assert q_boundary@q_boundary==0
xx=np.linspace(0,1,101)
assert np.allclose((1/2)*np.sin(np.pi*xx),(2/4)*np.sin(np.pi*xx))
A,kappa=2.,4.;flux=-A*np.pi;ratio=A/kappa
assert abs((-flux/np.pi)/ratio-kappa)<1e-12
print('端点不可辨识；两个参数的比值相同可产生相同状态；通量恢复通过')

E1 信息量S= 3.0 无噪声kappa= 2.0
E3 信息量S= 0.05396582105238862 无噪声kappa= 2.0
端点不可辨识；两个参数的比值相同可产生相同状态；通量恢复通过


## 题7：Fourier乘子、传感器反例与实际成本

直接验证单个模态在差分矩阵上的特征值，不调用主Notebook的谱求解函数。

In [4]:
M=64;k=4;j=np.arange(M);v=np.exp(2j*np.pi*k*j/M)
Av=M*M*(2*v-np.roll(v,1)-np.roll(v,-1))+v
lam=1+4*M*M*np.sin(np.pi*k/M)**2
assert np.max(abs(Av-lam*v))<1e-10
sensors8=np.arange(8)/8
assert np.max(abs(np.sin(8*np.pi*sensors8)))<1e-12
assert abs(np.sin(8*np.pi/16)-1)<1e-12
print('8个传感器完全看不见sin(8πx)，但它不是零函数。')
for r in P['operator']['cost']:print(r)
for r in P['operator']['amortization']:
    expected=r['quality_p95_1pct'] and r['denominator_s']>0
    assert (r['break_even_functions'] is not None)==expected
    print(r['network'],r['solver'],r['batch_functions'],r['reason'])
print('全部核验通过；不替代正文数学证明与人工审阅。')

8个传感器完全看不见sin(8πx)，但它不是零函数。
{'method': 'Spectral-32', 'batch_functions': 1, 'batch_s': 3.452600001764949e-05, 'per_function_s': 3.452600001764949e-05}
{'method': 'FD-32-to-64', 'batch_functions': 1, 'batch_s': 3.830600002402207e-05, 'per_function_s': 3.830600002402207e-05}
{'method': 'DeepONet', 'batch_functions': 1, 'batch_s': 0.0001436009999815724, 'per_function_s': 0.0001436009999815724}
{'method': 'Spectral-32', 'batch_functions': 200, 'batch_s': 0.00021683000013581478, 'per_function_s': 1.084150000679074e-06}
{'method': 'FD-32-to-64', 'batch_functions': 200, 'batch_s': 0.00022714999977324624, 'per_function_s': 1.1357499988662312e-06}
{'method': 'DeepONet', 'batch_functions': 200, 'batch_s': 0.00026203000015811995, 'per_function_s': 1.3101500007905997e-06}
{'method': 'Spectral-reused', 'batch_functions': 1, 'batch_s': 1.7480000024079346e-05, 'per_function_s': 1.7480000024079346e-05}
{'method': 'FD-reused', 'batch_functions': 1, 'batch_s': 1.6259000003628897e-05, 'per_function_s': 1